# DAF-17: Tune Without Touching the Test Days — TimeSeriesSplit Cross-Validation

## 📖 The story: Asha madam and the board-exam rule

Asha madam is the school coordinator. Every evening at 6 she decides: will the 600 children play outside tomorrow or not? Her model gives her tomorrow's PM2.5 number, and she turns it into "go out" or "stay in".

The model has **settings** she can change, like the knobs on a pressure cooker: `alpha` for Ridge, `max_depth` and `learning_rate` for the tree model. With default knobs, Ridge catches 109 of the 120 Poor days. She thinks: *"If I turn the knobs properly, it will catch more."*

So she turns a knob, checks the score on the **test days**, turns another knob, checks again. Twenty times. She keeps the best.

Sharma sir, the old maths teacher, laughs when he hears. *"Beta, this is a student who saw the board paper in advance and planned her study around it. She scores 95, but she is not a 95-marks student. She is a student who saw the paper. Good students write mock tests during the year, and in the right order: first Chapter 1 to 3, then test on Chapter 4. Never the Chapter 4 test after reading Chapter 5. That is cheating."*

Asha madam's 130 test days are the board paper. She has looked at it twenty times. It cannot tell her the truth any more.

## 🎯 The exact problem

> **How do we choose the best settings for the model, without ever looking at the test days, and without the mock tests mixing future days into the past?**

The answer is two tools. **`TimeSeriesSplit`** gives mock tests in the right order. **`RandomizedSearchCV`** tries many settings on those mock tests. The settings are written down before the board paper (the test days) is opened in DAF-18.

The pictures for this story are in `17_visual_walkthrough.ipynb`.

## What this ticket is about

Two changes to how we choose settings. The features and the question do not change.

```text
Before                                     After
settings = defaults                        settings chosen by cross-validation
(or chosen by looking at the test score)   on the 251 TRAIN days only
one score per model                        five scores per setting, averaged
```

The test period (130 days, from 1 Mar 2026) is not opened in this ticket. It is opened once, in DAF-18, with the settings frozen here.

## The new ideas

| Idea | One line | Where it comes from |
|---|---|---|
| Cross-validation | Cut the train days into pieces, fit on some, score on the next, repeat, average | sklearn |
| `TimeSeriesSplit` | The cut keeps time order: each fold trains on earlier rows, validates on the rows right after | `sklearn.model_selection` |
| Hyper-parameter | A setting you choose before fitting (`alpha`, `max_depth`, …). The model does not learn it | |
| `RandomizedSearchCV` | Try a fixed number of random settings, score each with cross-validation, keep the best | `sklearn.model_selection` |
| Frozen config | The chosen model and settings written to a file before DAF-18 | `models/final_config.json` |

Ordinary `KFold` with shuffling mixes future days into the training set, so it is not used here. The pictures are in `17_visual_walkthrough.ipynb`.

## What we search

| Model | Settings worth searching |
|---|---|
| Ridge | `alpha` |
| HistGradientBoosting | `learning_rate`, `max_depth`, `min_samples_leaf`, `max_iter` |
| RandomForest | `max_depth`, `min_samples_leaf`, `max_features` |

We tune **the best linear model and the best tree model** from DAF-15 and DAF-16. Ridge was best in DAF-16. Which tree model to carry forward is decided in Step 1, from the DAF-15 and DAF-16 scores, before any tuning runs.

Keep it small: 4 or 5 folds, a handful of settings per model, `scoring="neg_mean_absolute_error"`.

## Rules for this ticket

- **No test row anywhere.** The table is cut at 1 Mar 2026 and the test rows are dropped at the start. A code check shows it.
- **Preprocessing inside the fold.** The scaler is part of a `Pipeline`, so it is fitted on each fold's train rows only.
- **Compare by cross-validation score.** Tuned vs default is judged on the CV MAE, never on the test score.
- **Freeze before DAF-18.** The final model and settings go into a config file. After that, nothing changes.
- **Report honestly.** If tuning helps by a tiny amount, say so. The pictures already hint that Ridge may gain little.

## The plan, one step at a time

This is the proposed plan. Each step is added to this notebook only when we start it.

1. Rebuild the train table and prove no test row is in it. Choose the tree model to carry forward.
2. Build `TimeSeriesSplit` and plot the folds on a timeline.
3. Score both models at default settings with cross-validation. This is the baseline to beat.
4. Tune Ridge with `RandomizedSearchCV`.
5. Tune the tree model with `RandomizedSearchCV`.
6. Compare tuned vs default CV MAE for both models. Was it worth the complexity?
7. Choose the final model and write `models/final_config.json`.
8. Log the rows in `reports/experiments.csv`.

## Acceptance criteria (from the ticket)

- [ ] The fold plot shows every validation fold after its training fold
- [ ] No test row is used anywhere in this notebook — shown in code
- [ ] Tuned vs default CV MAE is reported for both models
- [ ] The final model and its settings are frozen before DAF-18 opens

## Explain back (answered at the end)

1. Why does `KFold` with shuffle cheat on this data?
2. How much did tuning help, and was it worth the complexity?